Cell 1: Mount Drive & Set Working Directory

In [ ]:
# Mount Google Drive to access the dataset stored in the cloud
from google.colab import drive
drive.mount('/content/drive')

# Switch to the project directory containing the datasets and the notebook
WORKING_DIR = '/content/drive/MyDrive/FTEC5560'
import os
os.chdir(WORKING_DIR)
print(f"Changed working directory to: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Changed working directory to: /content/drive/MyDrive/FTEC5560


Cell 2: Install Libraries

In [ ]:
!pip install openai numpy pandas scipy matplotlib seaborn

Cell 3: Import Libraries & Define Configuration

In [ ]:
import random
import numpy as np
import pandas as pd
from openai import OpenAI
from scipy.stats import norm
import os
import glob
from google.colab import userdata
import time # Import time module for response time measurement

# Retrieve the API Key from Colab Secrets for security
API_KEY_ENV = userdata.get('siliconflow_api_key')
BASE_URL = "https://api.siliconflow.cn/v1"

# Define the models, N-back difficulties, and temperature parameters to test
MODEL_NAMES_TO_TEST = ["deepseek-ai/DeepSeek-V3.2"] # Try DeepSeek-V3.2
N_BACK_VALUES = [1]
TEMPERATURES_TO_TEST = [0.7]
NUM_BLOCKS_PER_N = 1 # Use data from 10 blocks
TRIALS_PER_BLOCK = 30
# Specify the paths to the dataset folders
FIXED_DATASET_PATH_1BACK = "1back/"
FIXED_DATASET_PATH_2BACK = "2back/"

# Initialize the OpenAI client
client = OpenAI(api_key=API_KEY_ENV, base_url=BASE_URL)

Cell 4: Helper Functions

In [ ]:
# Define the NEW number of trials per block based on your actual dataset
NEW_TRIALS_PER_BLOCK = 24

def load_sequence_and_targets_from_file(file_path):
    """Load the letter sequence and target labels from a specified file."""
    with open(file_path, 'r') as f:
        content = f.read().strip() # Remove leading/trailing whitespace, including the newline
    print(f"DEBUG: File {file_path}, Content Length: {len(content)}") # Debug print
    print(f"DEBUG: Content (repr): {repr(content)}") # Debug print

    # Calculate split point based on the new trial count
    split_point = NEW_TRIALS_PER_BLOCK
    sequence_part = content[:split_point]
    targets_part = content[split_point:]

    print(f"DEBUG: Sequence Part Length: {len(sequence_part)}, Targets Part Length: {len(targets_part)}") # Debug print

    sequence = list(sequence_part.upper())
    targets = list(targets_part.lower())
    return sequence, targets

def get_model_response(client, current_stimulus, model_name, temperature=0.7):
    """
    Call the model API to get a response for the current stimulus letter.
    Mimics the original code's approach: sends the letter, parses first character strictly.
    """
    # Create the initial system message (like the original)
    system_message = {
        "role": "system",
        "content": "You are asked to perform a 1-back task. You will see a sequence of letters. Your task is to respond with 'm' (no quotation marks, just the letter m) whenever the current letter is the same as the previous one, and '-' (no quotation marks, just the dash sign) otherwise. Only 'm' and '-' are allowed responses. No explanations needed: please don't output any extra words!! The sequence will be presented one letter at a time. Now begins the task."
    }
    # Prepare the conversation history for the API call
    messages = [
        {"role": "system", "content": system_message["content"]},
        {"role": "user", "content": current_stimulus}
    ]

    start_time = time.time()

    completion = client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=10,
        temperature=temperature,
    )

    response_time = time.time() - start_time

    response_text = completion.choices[0].message.content.strip().lower()
    print(f"      Stimulus: '{current_stimulus}', Raw API Response: '{response_text}', RT: {response_time:.2f}s")

    if not response_text:
        print("        WARNING: Empty response received.")
        return '-', response_time

    first_char = response_text[0]

    if first_char == 'm':
        return 'm', response_time
    elif first_char in ['-', 'n']:
        return '-', response_time
    else:
        print(f"        RULE VIOLATION! Extracting the first letter of the response: '{first_char}'")
        if first_char == 'm':
            return 'm', response_time
        elif first_char == '-':
            return '-', response_time
        else:
            raise ValueError(f"Invalid response character '{first_char}'. Please respond with 'm' or '-'. Full response was: '{response_text}'")

def calculate_metrics_per_block(targets, responses):
    """Calculate psychophysical metrics for a single block based on targets and responses."""
    if len(targets) != len(responses):
        print(f"DEBUG: calculate_metrics_per_block called with mismatch. Targets: {len(targets)}, Responses: {len(responses)}")
        raise ValueError("Targets and responses must have the same length.")

    hits = sum(1 for t, r in zip(targets, responses) if t == 'm' and r == 'm')
    misses = sum(1 for t, r in zip(targets, responses) if t == 'm' and r == '-')
    fas = sum(1 for t, r in zip(targets, responses) if t == '-' and r == 'm')
    crs = sum(1 for t, r in zip(targets, responses) if t == '-' and r == '-')

    total_signal = hits + misses
    total_noise = fas + crs

    hit_rate = hits / total_signal if total_signal > 0 else 0.0
    fa_rate = fas / total_noise if total_noise > 0 else 0.0

    accuracy = (hits + crs) / len(targets) if targets else 0.0

    epsilon = 1e-9
    corrected_hr = np.clip(hit_rate, epsilon, 1 - epsilon)
    corrected_far = np.clip(fa_rate, epsilon, 1 - epsilon)
    d_prime = norm.ppf(corrected_hr) - norm.ppf(corrected_far)

    return {
        'hit_rate': hit_rate, 'fa_rate': fa_rate,
        'accuracy': accuracy, 'd_prime': d_prime,
        'hits': hits, 'misses': misses, 'fas': fas, 'crs': crs
    }

def run_single_condition(n_back, num_blocks, client, model_name, temp_value, condition_label):
    """Run a single experimental condition (model, N-back, temp) for a specified number of blocks."""
    dataset_path = FIXED_DATASET_PATH_1BACK if n_back == 1 else FIXED_DATASET_PATH_2BACK

    results = []
    all_files = sorted(glob.glob(os.path.join(dataset_path, "*.txt")))
    selected_files = all_files[:num_blocks]

    print(f"Starting {condition_label}...")
    for idx, file_path in enumerate(selected_files):
        block_id = int(os.path.basename(file_path).split('.')[0])
        print(f"  Processing Block {idx+1}/{num_blocks} from {file_path.split('/')[-1]}...")

        sequence, targets = load_sequence_and_targets_from_file(file_path) # This will now print debug info
        responses = []
        response_times = []

        for i in range(len(sequence)):
            current_letter = sequence[i]
            prompt = current_letter

            try:
                resp, rt = get_model_response(client, prompt, model_name, temperature=temp_value)
                responses.append(resp)
                response_times.append(rt)
            except ValueError as e:
                print(f"ValueError on trial {i} (stimulus '{prompt}'): {e}")
                responses.append('-')
                response_times.append(np.nan)
                continue
            except Exception as e:
                print(f"General Error on trial {i} (stimulus '{prompt}'): {type(e).__name__} - {e}")
                responses.append('-')
                response_times.append(np.nan)
                time.sleep(1)
                continue

        print(f"DEBUG: Before calculate_metrics - Targets Len: {len(targets)}, Responses Len: {len(responses)}") # Debug print

        # Calculate metrics for this specific block
        # Ensure lengths match before calculating, though they should now always match due to the catch-all
        if len(responses) != len(targets):
             print(f"Warning: Length mismatch for block {block_id}. Targets: {len(targets)}, Responses: {len(responses)}. Skipping block.")
             continue # Skip this block if there's still a mismatch despite our fixes

        block_metrics = calculate_metrics_per_block(targets, responses)
        block_metrics['block_id'] = block_id
        block_metrics['model'] = model_name
        block_metrics['n_back'] = n_back
        block_metrics['temperature'] = temp_value
        block_metrics['avg_rt'] = np.mean([t for t in response_times if not np.isnan(t)])

        results.append(block_metrics)
        print(f"    Completed Block {block_id}. Accuracy: {block_metrics['accuracy']:.3f}, d': {block_metrics['d_prime']:.3f}")

    print(f"Completed {condition_label}. Processed {len(results)} blocks.\n")
    return pd.DataFrame(results)

Cell 5: Define Main Experiment Runner

In [ ]:
# Main loop to iterate through all defined conditions (models, N-back values, temperatures)
all_results_df = pd.DataFrame()

for model_name in MODEL_NAMES_TO_TEST:
    for n_val in N_BACK_VALUES:
        for temp_val in TEMPERATURES_TO_TEST:
            condition_label = f"{model_name.split('/')[-1]}_Fixed_T{temp_val}_{n_val}back"
            df = run_single_condition(
                n_back=n_val,
                num_blocks=NUM_BLOCKS_PER_N,
                client=client,
                model_name=model_name,
                temp_value=temp_val,
                condition_label=condition_label
            )
            all_results_df = pd.concat([all_results_df, df], ignore_index=True)

# Save the final aggregated results to a CSV file
all_results_df.to_csv("llm_nback_results.csv", index=False)
print("Final DataFrame shape:", all_results_df.shape)
print(all_results_df.head())
print("\nResults saved to llm_nback_results.csv")

Starting DeepSeek-V3.2_Fixed_T0.7_1back...
  Processing Block 1/1 from 0.txt...
DEBUG: File 1back/0.txt, Content Length: 49
DEBUG: Content (repr): 'qvbvvvfxxxqrrhbfffnbppyg\n----mm--mm--m---mm---m--'
DEBUG: Sequence Part Length: 24, Targets Part Length: 25
      Stimulus: 'Q', Raw API Response: '-', RT: 2.28s
      Stimulus: 'V', Raw API Response: '-', RT: 2.13s
      Stimulus: 'B', Raw API Response: '-', RT: 1.41s
      Stimulus: 'V', Raw API Response: '-', RT: 2.22s
      Stimulus: 'V', Raw API Response: '-', RT: 2.88s
      Stimulus: 'V', Raw API Response: '-', RT: 1.35s
      Stimulus: 'F', Raw API Response: '-', RT: 1.22s
      Stimulus: 'X', Raw API Response: '-', RT: 2.39s
      Stimulus: 'X', Raw API Response: '-', RT: 1.57s
      Stimulus: 'X', Raw API Response: '-', RT: 2.65s
      Stimulus: 'Q', Raw API Response: '-', RT: 1.02s
      Stimulus: 'R', Raw API Response: '-', RT: 0.87s
      Stimulus: 'R', Raw API Response: '-', RT: 0.86s
      Stimulus: 'H', Raw API Response: '-

Cell 6: Execute Full Experiment & Save Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Load results if needed (or use the existing all_results_df)
results_df = all_results_df.copy()

# Check if the DataFrame has the required columns before proceeding
required_cols = ['model', 'n_back', 'accuracy', 'd_prime']
missing_cols = [col for col in required_cols if col not in results_df.columns]

if missing_cols:
    print(f"Error: Missing required columns in results DataFrame: {missing_cols}")
    print(f"Available columns: {list(results_df.columns)}")
    print("Cannot proceed with analysis.")
else:
    # Calculate summary statistics grouped by model and n_back
    summary_stats = results_df.groupby(['model', 'n_back'])[['accuracy', 'd_prime']].agg(['mean', 'std']).round(4)
    print("Summary Statistics (Mean and Std for Accuracy and d_prime):\n")
    print(summary_stats)

    # --- Visualization ---
    sns.set_style("whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Plot 1: Accuracy by Model and N-back
    sns.barplot(data=results_df, x='model', y='accuracy', hue='n_back', ax=axes[0], errorbar=('ci', 95))
    axes[0].set_title('Accuracy by Model and N-back Condition')
    axes[0].tick_params(axis='x', rotation=45)

    # Plot 2: d-prime by Model and N-back
    sns.barplot(data=results_df, x='model', y='d_prime', hue='n_back', ax=axes[1], errorbar=('ci', 95))
    axes[1].set_title('Sensitivity (d-prime) by Model and N-back Condition')
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    # Also plot by N-back, grouped by model (alternative view)
    if 'n_back' in results_df.columns and 'model' in results_df.columns: # Double-check again for plotting
        fig2, axes2 = plt.subplots(1, 2, figsize=(14, 6))

        # Plot 3: Accuracy by N-back and Model
        sns.barplot(data=results_df, x='n_back', y='accuracy', hue='model', ax=axes2[0], errorbar=('ci', 95))
        axes2[0].set_title('Accuracy by N-back Condition and Model')

        # Plot 4: d-prime by N-back and Model
        sns.barplot(data=results_df, x='n_back', y='d_prime', hue='model', ax=axes2[1], errorbar=('ci', 95))
        axes2[1].set_title('Sensitivity (d-prime) by N-back Condition and Model')

        plt.tight_layout()
        plt.show()
    else:
        print("Could not generate plots due to missing grouping columns.")

Error: Missing required columns in results DataFrame: ['model', 'n_back', 'accuracy', 'd_prime']
Available columns: []
Cannot proceed with analysis.
